# TRACE-RAG — end-to-end run on Colab (GPU)

This notebook runs the whole stack on real data with a real LLM, so the
defence can be exercised end to end:

1. **ingest** a BEIR corpus with simulated contributors (`SourceAssigner`);
2. **index** it (this is the GPU step);
3. **poison** the passage that actually carries the gold answer, with Person C's
   attack helpers (entity swap / negation), ingested as a burst from one fresh
   attacker source;
4. **replay** a Zipf-repeated query stream through the full pipeline with
   Person B's trust layer wired in (`TrustLedger` + `CorroborationVerifier` +
   `TrustPolicy` + `VerificationQueue`);
5. **write** `metrics.json`, `stream.jsonl`, `trust_history.csv` and the B6
   figures.

What this is **not**: Person C's headline evaluation (ASR, V0–V5 ablations,
baselines, confidence intervals) lives in Person C's runner. The numbers here
are a component smoke test on real data and the script says so in its output.

| Step | 200k-passage subset on a T4 | full 2.68M corpus |
|---|---|---|
| install | 1–3 min | 1–3 min |
| download | 2–5 min | 5–15 min |
| ingest | 1–3 min | 10–20 min |
| index (Contriever) | 8–20 min | 2–4 h, needs IVF-PQ |
| stream + trust | 5–20 min | same |

**Runtime → Change runtime type → GPU before running anything.**

## 0. Runtime check

Colab ships a CUDA build of PyTorch. This cell proves the GPU is visible.

In [ ]:
!nvidia-smi -L
import torch, sys
print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu: {props.name}  vram: {props.total_memory / 1e9:.1f} GB")

## 1. Get the code and install

Public repo: the clone below works as is. Private fork: use
`https://<TOKEN>@github.com/...` instead (a GitHub personal access token with
`repo` scope), and never paste the token into a cell you save.

In [ ]:
%%bash
set -e
REPO=/content/trace-rag
if [ -d "$REPO/.git" ]; then
  git -C "$REPO" pull --ff-only
else
  git clone --depth 1 "https://github.com/vaishnavissl1/-Adaptive-Trust-Based-Framework-for-Detecting-and-Preventing-Knowledge-Base-Poisoning-in-RAG-Systems.git" "$REPO"
fi
cd "$REPO"
pip install -q -e ".[all,data]"
python scripts/check_setup.py

## 2. Download the data

`--subset 200000` builds a 200k-passage development corpus that **keeps every
gold passage** for 500 sampled test queries — a random slice would drop the gold
passages and make retrieval look broken for the wrong reason. The download is
resumable, so re-running the cell after a disconnect is safe.

The full 2.68M-passage corpus is one flag away (no `--subset`), but it needs
`config/full_nq.yaml` (IVF-PQ index) and roughly 2–4 hours of indexing on a T4.

In [ ]:
%%bash
set -e
cd /content/trace-rag
python scripts/download_data.py --dataset nq --out data/nq --subset 200000 --queries 500
ls -lh data/nq | head

## 3. The answer model: Ollama

Ollama runs quantised 7–8B models inside Colab's GPU (a T4 has 16 GB; a q4 7B
model needs ~5 GB). `qwen2.5:7b` is Apache-2.0 and needs no approval;
`llama3.1:8b` works too if you prefer it.

Alternatives, if you would rather not use Ollama:

* vLLM — `pip install vllm` then `vllm serve Qwen/Qwen2.5-7B-Instruct --port 8000`,
  and swap the `--set` flags below for
  `--set generation.backend=openai --set generation.base_url=http://localhost:8000/v1`;
* in-process transformers — `--set generation.backend=huggingface`
  (slowest, and any 7B model in bf16 is a tight fit on a T4).

After a runtime restart the daemon is gone: re-run this cell.

In [ ]:
%%bash
set -e
if ! command -v ollama >/dev/null 2>&1; then
  curl -fsSL https://ollama.com/install.sh | sh
fi
# Colab has no systemd, so start the daemon ourselves and wait for the API.
if ! curl -sf http://127.0.0.1:11434/api/tags >/dev/null 2>&1; then
  nohup ollama serve >/tmp/ollama.log 2>&1 &
fi
for i in $(seq 1 30); do
  curl -sf http://127.0.0.1:11434/api/tags >/dev/null 2>&1 && break
  sleep 2
done
ollama pull qwen2.5:7b        # ~4.7 GB, downloaded once per session
ollama list

Sanity-check the model through the project's own backend before the long run:

In [ ]:
import sys
sys.path.insert(0, "/content/trace-rag/src")
from trace_rag.generation.llm import OllamaLLM

llm = OllamaLLM("qwen2.5:7b")
reply = llm.generate("Answer in one short sentence: who designed the Eiffel Tower?", max_tokens=48)
print("model:", llm.name, "| latency:", round(reply.latency_ms), "ms")
print("reply:", reply.text)

## 4. Quick validation run (~5–10 min)

20k passages, 2 targets, 6 stream steps. This is the fast check that ingest,
index, poisoning, retrieval, verification and the figures all work on this
machine **before** paying for the full index.

Watch for: the poison is crafted from a passage that carries the gold answer
(`chosen_by` says which rule picked it — with `--limit` the qrels gold passage
may not be in the head of the file, and then the runner falls back to the
retrieved pool), it is retrieved for its target question, the verifier returns a
verdict for it, and its trust falls (`poison_trust`).

In [ ]:
%%bash
cd /content/trace-rag
python scripts/run_trust_stream.py \
  --config config/nq_gpu.yaml \
  --corpus data/nq/corpus_subset.parquet \
  --queries data/nq/queries_subset.jsonl \
  --qrels data/nq/qrels.tsv \
  --limit 20000 --steps 6 --targets 2 \
  --out runs/nq/quick \
  --set generation.backend=ollama --set generation.model_name=qwen2.5:7b

## 5. Full run: the whole 200k-passage subset

Same script, no `--limit`, 40 stream steps and 5 poisoned target questions (five
documents from one fresh source inside a minute is also what makes the burst
detector fire, so the cold-start discount is exercised).

Expect ~25–45 min on a T4: ingest (~2 min) + index (~15 min) + the stream, where
each verifier call is an Ollama generation.

Re-running later in the same session: pass `--skip-index` **with the same
`--out`** to reuse the saved index instead of re-embedding the corpus.

In [ ]:
%%bash
cd /content/trace-rag
python scripts/run_trust_stream.py \
  --config config/nq_gpu.yaml \
  --corpus data/nq/corpus_subset.parquet \
  --queries data/nq/queries_subset.jsonl \
  --qrels data/nq/qrels.tsv \
  --steps 40 --targets 5 \
  --out runs/nq/trust_stream \
  --set generation.backend=ollama --set generation.model_name=qwen2.5:7b

## 6. Read the results

In [ ]:
import json, pathlib

base = pathlib.Path("/content/trace-rag/runs/nq/trust_stream")
metrics = json.loads((base / "metrics.json").read_text())

keys = ("bands", "llm", "verifier_mode", "steps", "target_steps", "poison_documents",
        "poison_retrieved_rate", "poison_citation_rate", "poison_quarantined",
        "clean_false_quarantine", "first_quarantine_step", "queued_verifications",
        "mean_llm_calls_per_query", "mean_latency_ms", "verifications")
for key in keys:
    print(f"{key:>26}: {metrics[key]}")

print("\npoison planted:")
for t in metrics["poison_targets"]:
    print(f"  {t['doc_id']:<24} {t['attack']:<18} <- {t['origin']}  ({t['chosen_by']})")

print("\npoison state after the stream:")
for doc_id, snap in metrics["poison_trust"].items():
    print(f"  {doc_id:<24} t_eff={snap['t_eff']:.3f}  {snap['status']}")

rows = [json.loads(line) for line in (base / "stream.jsonl").read_text().splitlines()]
print("\nsteps where a poison was retrieved:")
for row in rows:
    if row["poison_retrieved"]:
        verdicts = ", ".join(f"{doc}->{out}" for doc, out in row.get("verified_docs", []))
        print(f"  step {row['step']:>2} {row['query'][:42]:<42} "
              f"cited_poison={row['poison_cited']} bands={row['bands']}")
        print(f"           verdicts: {verdicts or 'none'}")

In [ ]:
from IPython.display import Image, display, Markdown
import pathlib

figures = pathlib.Path("/content/trace-rag/runs/nq/trust_stream/figures")
names = sorted(p.name for p in figures.glob("*.png"))
print(f"{len(names)} figures written")

def show(token: str, if_empty: str) -> None:
    hits = [name for name in names if token in name]
    if not hits:
        print(f"  (no figure matching {token!r}: {if_empty})")
        return
    for name in hits:
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(figures / name)))

show("poison", "no poisoned chunk was tracked")
show("attacker", "the attacker source earned no trust update: nothing was refuted")
show("quarantine", "nothing was blocked in this run, so there is no timeline")


## 7. What the numbers mean, and what they do not

* **`poison_quarantined` / `t_eff` / `pollution` of the answer** are the defence
  working: the poison is retrieved (that is the attack landing) and the verifier
  then refutes it, so its trust decays and the retriever stops surfacing it.
* **`clean_false_quarantine`** is the cost side: honest passages blocked by
  mistake. It should stay at 0 on this stream.
* **Only escalated passages get verified.** Which passages escalate is Person A's
  detector (heuristic fallback until a labelled scorer is trained). If the
  poison never escalates here, the trust layer never sees it — that is a
  detection-coverage statement, not a trust-layer failure, and the fix is a
  trained scorer, not a lower threshold.
* **`mean_llm_calls_per_query`** is the budget: one answer call per query, plus
  at most two per verified passage (claim extraction + the leave-one-out
  counterfactual).
* The stream's poison is generated by the runner itself; Person C's ASR
  numbers over the full attack set come from Person C's runner.
* Do **not** quote numbers from `--force-medium` runs. Measured on the bundled
  mini corpus: forcing every passage to MEDIUM also verifies off-topic passages
  that real banding keeps at LOW, and those were refuted and quarantined by
  mistake. With the real bands the poison is escalated because it *is*
  suspicious, and honest passages are only affected when the detector escalates
  them too.

## 8. Optional: ablations and stricter settings

Each of these re-runs the **doc-only V2 ablation** (`--no-hierarchical`: no
family/source prior, no trust inheritance) and a strict single-refutation
quarantine. They re-ingest and re-index, so budget another ~20 min each, or drop
`--limit 20000` runs if the full one is enough.

In [ ]:
%%bash
cd /content/trace-rag

echo "=== V2 ablation: document-only trust (no family/source prior) ==="
python scripts/run_trust_stream.py \
  --config config/nq_gpu.yaml \
  --corpus data/nq/corpus_subset.parquet \
  --queries data/nq/queries_subset.jsonl --qrels data/nq/qrels.tsv \
  --limit 20000 --steps 12 --targets 3 --no-hierarchical \
  --out runs/nq/stream_doc_only \
  --set generation.backend=ollama --set generation.model_name=qwen2.5:7b

echo "=== strict: one refutation quarantines ==="
python scripts/run_trust_stream.py \
  --config config/nq_gpu.yaml \
  --corpus data/nq/corpus_subset.parquet \
  --queries data/nq/queries_subset.jsonl --qrels data/nq/qrels.tsv \
  --limit 20000 --steps 12 --targets 3 --quarantine-refutations 1 \
  --out runs/nq/stream_strict \
  --set generation.backend=ollama --set generation.model_name=qwen2.5:7b

## 9. Troubleshooting

| Symptom | Fix |
|---|---|
| `cuda available: False` | Runtime → Change runtime type → GPU, then rerun cell 0 |
| `cannot reach http://localhost:11434` | re-run the Ollama cell (a runtime restart kills the daemon) |
| `ollama pull` very slow | it is a ~5 GB download; `qwen2.5:3b` is a lighter fallback |
| `CUDA out of memory` while indexing | `--set embedding.batch_size=32` (or 16) |
| `CUDA out of memory` from the model | use a smaller quantisation/model (`qwen2.5:3b`) — Ollama and the embedder share the GPU |
| download stopped mid-way | re-run the download cell; it resumes |
| want the full 2.68M corpus | no `--subset` on the download, `--config config/full_nq.yaml`, expect hours and ~8 GB RAM for the index |
| index seems empty after a restart | re-run `--skip-index`-free, or keep `--out` the same and pass `--skip-index` |
| poison ends MONITORED, not QUARANTINED | it was refuted once: repeated target queries are what reach the second refutation. More `--steps`, or `--quarantine-refutations 1` to show the strict setting |